In [ ]:
import os
from pathlib import Path
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp
from rich.console import Console
from rich.table import Table

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "tdmd").exists() else cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tdmd import FFTTransform, TDMDII

plt.style.use("ggplot")


In [ ]:
x = jnp.linspace(-1.0, 1.0, 96)
y = jnp.linspace(-1.0, 1.0, 96)
t = jnp.linspace(0.0, 2.0 * jnp.pi, 128)
X, Y, T = jnp.meshgrid(x, y, t, indexing="ij")

Xf = 5.0 * X - 5.0
Yf = 5.0 * Y - 5.0
R = jnp.sqrt(Xf**2 + Yf**2)
Theta = jnp.arctan2(Yf, Xf + 1.0e-6)

Z = (
    7.0 * jnp.sin(1.7 * Xf + 0.9 * T)
    + 6.0 * jnp.cos(1.3 * Yf - 1.2 * T)
    + 5.0 * jnp.sin(0.9 * (Xf + Yf) + 0.7 * T)
    + 4.0 * jnp.cos(2.2 * R - 1.8 * T)
    + 3.0 * jnp.sin(3.0 * Theta + 0.6 * R - 1.1 * T)
    + 2.5 * jnp.cos(1.8 * (Xf - Yf) + 0.3 * R + 0.8 * T)
)


In [ ]:
train_tensor = jnp.transpose(Z[:, :, :96], (0, 2, 1))
pred_idx = train_tensor.shape[1]
pred_steps = np.arange(pred_idx + 1)


In [ ]:
gamma = 0.999
signvals_threshold = 1.0e-8
model = TDMDII(FFTTransform(), gamma=gamma, signvals_threshold=signvals_threshold)
model.fit(train_tensor)

pred_tensor = np.asarray(model.forecast(pred_idx + 1)).real
state_errors = np.array([
    np.linalg.norm(np.asarray(Z[:, :, step]) - pred_tensor[:, step, :]) / np.linalg.norm(np.asarray(Z[:, :, step]))
    for step in pred_steps
])

snapshot_true = np.asarray(Z[:, :, pred_idx])
snapshot_pred = np.asarray(model.predict_step(pred_idx)).real
snapshot_re = float(state_errors[pred_idx])

table = Table(title="Plasma TDMDII")
table.add_column("Name", no_wrap=True)
table.add_column("Value")
table.add_row("sequence shape", f"{tuple(Z.shape)}")
table.add_row("k_max", f"{int(model.multirank.max())}")
table.add_row("sum k_j", f"{int(model.multirank.sum())}")
table.add_row("mean state-wise RE", f"{state_errors.mean():.4e}")
table.add_row("max state-wise RE", f"{state_errors.max():.4e}")
table.add_row("snapshot RE", f"{snapshot_re:.4e}")
Console().print(table)


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(23, 4.8), constrained_layout=True)

axes[0].imshow(np.asarray(Z[:, :, 0]), origin="lower", cmap="magma")
axes[0].set_title("Snapshot t=0")
axes[0].set_xticks([])
axes[0].set_yticks([])

axes[1].imshow(snapshot_true, origin="lower", cmap="magma")
axes[1].set_title(f"Target t={pred_idx}")
axes[1].set_xticks([])
axes[1].set_yticks([])

axes[2].imshow(snapshot_pred, origin="lower", cmap="magma")
axes[2].set_title("Reconstruction")
axes[2].set_xticks([])
axes[2].set_yticks([])

residual_img = axes[3].imshow(snapshot_true - snapshot_pred, origin="lower", cmap="coolwarm")
axes[3].set_title("Residual")
axes[3].set_xticks([])
axes[3].set_yticks([])
fig.colorbar(residual_img, ax=axes[3], shrink=0.9, pad=0.02)

axes[4].plot(pred_steps, state_errors, linewidth=2, color="tab:green")
axes[4].axvline(pred_idx, color="black", linestyle="--", linewidth=1)
axes[4].set_title("RE vs Snapshot")
axes[4].set_xlabel("Snapshot")
axes[4].set_ylabel("RE")
axes[4].grid(True, alpha=0.3)

plt.show()
